<a href="https://colab.research.google.com/github/ssenaozz/LinkTree/blob/master/Methods_Codebook.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import pandas as pd
import itertools
from collections import Counter
import re

# 1. Dosya Adı
file_name = 'zeeschuimer-export-tiktok.com-2026-03-29T201110(1).csv'

print(f"--- İşlem Başlatılıyor: {file_name} ---")

try:
    # 2. Veriyi Okuma
    df = pd.read_csv(file_name, on_bad_lines='skip', encoding='utf-8')
    print(f"Veri yüklendi. Satır sayısı: {len(df)}")

    all_edges = []
    all_nodes_list = []

    # 3. Hashtag Ayıklama Fonksiyonu
    def process_hashtags(row):
        tags_raw = str(row['data/desc'])
        if tags_raw == 'nan' or tags_raw == '':
            return

        # Regex ile hashtagleri ayıkla
        hashtags = re.findall(r'#(\w+)', tags_raw)
        tags = [t.strip().lower() for t in hashtags if t.strip()]
        unique_tags = sorted(set(tags))

        # Edges (Bağlar) için çiftleri ekle
        all_edges.extend(list(itertools.combinations(unique_tags, 2)))

        # Nodes (Düğümler) için tüm tagleri listeye ekle (Sıklık hesaplamak için)
        all_nodes_list.extend(tags)

    print("Veriler işleniyor (Hashtagler ayıklanıyor)...")
    for index, row in df.iterrows():
        process_hashtags(row)

    # 4. EDGES (Kenarlar) Listesini Oluşturma
    edge_counts = Counter(all_edges)
    edges_df = pd.DataFrame(
        [{'Source': e[0], 'Target': e[1], 'Weight': count} for e, count in edge_counts.items()]
    )
    # Filtreleme (Weight > 1) - Modülerlik için kritik
    filtered_edges = edges_df[edges_df['Weight'] > 1]

    # 5. NODES (Düğümler) Listesini Oluşturma
    node_counts = Counter(all_nodes_list)
    nodes_df = pd.DataFrame(
        [{'Id': node, 'Label': node, 'Count': count} for node, count in node_counts.items()]
    )

    # Sadece Edges listesinde kalan düğümleri Nodes listesinde tutalım (Temizlik için)
    # Filtrelenmiş kenarlardaki tüm kaynak ve hedef düğümleri al
    active_nodes = set(filtered_edges['Source']).union(set(filtered_edges['Target']))
    filtered_nodes = nodes_df[nodes_df['Id'].isin(active_nodes)]

    # 6. Kaydetme
    edge_output = 'tiktok_gephi_edge_list.csv'
    node_output = 'tiktok_gephi_node_list.csv'

    filtered_edges.to_csv(edge_output, index=False)
    filtered_nodes.to_csv(node_output, index=False)

    print(f"\n--- BAŞARILI ---")
    print(f"Edges Dosyası: {edge_output} ({len(filtered_edges)} bağ)")
    print(f"Nodes Dosyası: {node_output} ({len(filtered_nodes)} düğüm)")
    print("Gephi'de önce Nodes, sonra Edges dosyasını içe aktarabilirsiniz.")

except FileNotFoundError:
    print(f"HATA: '{file_name}' bulunamadı.")
except Exception as e:
    print(f"HATA: {e}")

--- İşlem Başlatılıyor: zeeschuimer-export-tiktok.com-2026-03-29T201110(1).csv ---
Veri yüklendi. Satır sayısı: 116
Veriler işleniyor (Hashtagler ayıklanıyor)...

--- BAŞARILI ---
Edges Dosyası: tiktok_gephi_edge_list.csv (287 bağ)
Nodes Dosyası: tiktok_gephi_node_list.csv (65 düğüm)
Gephi'de önce Nodes, sonra Edges dosyasını içe aktarabilirsiniz.
